In [1]:
import torch  
import torch.nn as nn  
import torchvision.models as models  
  
# 假设我们有一个预训练的ResNet模型  
model = models.resnet18(pretrained=True)  
  
# 定义一个辅助函数来计算卷积层中每个过滤器的L1范数  
def compute_l1_norm(layer):  
    if isinstance(layer, nn.Conv2d):  
        return torch.norm(layer.weight.data, 1, dim=[1, 2, 3])  
    else:  
        return None  
  
# 定义一个函数来剪枝模型中的卷积层  
def prune_model(model, pruning_rate=0.2):  
    for name, module in model.named_modules():  
        if isinstance(module, nn.Conv2d):  
            l1_norms = compute_l1_norm(module)  
            # 按L1范数排序  
            sorted_indices = torch.argsort(l1_norms, descending=True)  
              
            # 计算需要剪枝的过滤器数量  
            num_filters = module.weight.data.size(0)  
            num_to_prune = int(pruning_rate * num_filters)  
              
            # 保留重要的过滤器，将不重要的过滤器权重设为0（结构化剪枝）  
            prune_indices = sorted_indices[num_filters - num_to_prune:]  
            module.weight.data = module.weight.data[prune_indices]  
              
            # 如果存在偏置项，也需要相应地剪枝  
            if module.bias is not None:  
                module.bias.data = module.bias.data[prune_indices]  
              
            # 注意：这里我们直接修改了权重，但实际应用中可能需要更复杂的处理，如添加mask等  
              
            # 打印剪枝信息（可选）  
            print(f'Pruning layer {name}, pruning rate: {pruning_rate:.2%}, '  
                  f'original filters: {num_filters}, pruned filters: {num_filters - num_to_prune}')  
  
# 应用剪枝函数到模型上  
prune_model(model, pruning_rate=0.2)  
  
# 注意：剪枝后的模型可能需要进行微调以恢复性能  
# ...（这里省略了微调代码）  
  
# 剪枝后的模型将具有更少的参数，但请注意，直接设置权重为0并不会减少模型的计算量  
# 在实际部署时，可能需要额外的步骤来去除这些零权重参数，如使用稀疏矩阵库或自定义前向传播函数

d:\anaconda\envs\pt\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\anaconda\envs\pt\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\87486/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
 69%|██████▊   | 30.6M/44.7M [09:53<04:31, 54.1kB/s]  


RuntimeError: invalid hash value (expected "f37072fd", got "4b2aaa2989c195059120ba9e73603c865091d27559675aa405b62a6f1999e6d1")